# Agentic Trading Desk: test drive

Try every new piece yourself before the Space goes public: the MCP servers one by one, the risk
desk, the research library, and finally one trader's full session.

- Everything here writes to a separate folder, `data_test/`, so **your real traders are never touched**.
- Cells marked **free** call no language model. Cells marked **paid** do, and say roughly how much.
- Run the cells in order. Section 1 is needed by all the others.

## 1. Setup (free)

In [ ]:
# Point the project at a scratch data folder. This must happen before any backend import.
import os
os.environ["DATA_DIR"] = "data_test"
os.environ["SCHEDULER_ENABLED"] = "false"
os.environ["USE_BROWSER"] = "true"

In [ ]:
# On Windows, a stdio MCP server started from a Jupyter kernel crashes writing to stderr.
# The fix: send the server's stderr to the null device, so it always has somewhere to write.
import functools
import subprocess
import agents.mcp.server

agents.mcp.server.stdio_client = functools.partial(agents.mcp.server.stdio_client, errlog=subprocess.DEVNULL)

In [ ]:
import json
from contextlib import AsyncExitStack
from IPython.display import Markdown, display

from agents import Agent, Runner, add_trace_processor, trace
from backend import config, market
from backend.accounts import Account
from backend.database import rows
from backend.mcp_servers import _own, memory_server, start_shared_servers
from backend.reset import reset_traders
from backend.traders import trader_model

reset_traders()  # four fresh $10,000 accounts in data_test/
print("Data folder :", config.DATA_DIR)
print("Prices      :", "Massive, previous close" if market.price_is_delayed() else "Massive, live" if market.massive_api_key else "simulated")
print("Models      :", {name: trader_model(name) for name in config.NAMES})
print("Risk manager:", config.RISK_MODEL)

In [ ]:
async def call(server, tool, **arguments):
    """Call one MCP tool and print what came back."""
    try:
        result = await server.call_tool(tool, arguments)
        text = " ".join(getattr(part, "text", "") for part in result.content)
    except Exception as error:
        text = f"ERROR: {error}"
    try:
        text = json.dumps(json.loads(text), indent=2)
    except ValueError:
        pass
    print(f"{tool}({', '.join(f'{k}={v!r}' for k, v in arguments.items())})")
    print(text[:1400] + ("\n..." if len(text) > 1400 else ""), "\n")

## 2. The MCP servers, one at a time (free)

Each cell starts one server, lists its tools and calls them. This is exactly what the agents see.

### Market analytics
Prices, daily history, indicators and a comparison with the S&P 500. On the free Massive plan a
symbol's first history request can answer "rate limit reached": wait a minute and run the cell again.

In [ ]:
async with _own("market_server") as server:
    print("Tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "lookup_share_price", symbol="NVDA")
    await call(server, "get_indicators", symbol="NVDA")
    await call(server, "compare_to_benchmark", symbol="NVDA", days=90)
    await call(server, "get_price_history", symbol="NVDA", days=10)
    await call(server, "lookup_share_price", symbol="NOTREAL")

### Fundamentals and filings
A company profile from Massive, and reported financials and filings from SEC EDGAR.

In [ ]:
async with _own("fundamentals_server") as server:
    print("Tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "get_company_profile", symbol="MSFT")
    await call(server, "get_financials", symbol="MSFT")
    await call(server, "list_recent_filings", symbol="MSFT")

### Research library (Qdrant vector memory)
Store two notes, then search with **different words**. The right note should come back, because the
search is by meaning. The first run loads the embedding model, which takes a few seconds.

In [ ]:
async with _own("library_server") as server:
    print("Tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "store_research", trader="Warren", topic="Costco membership renewals",
               symbols=["COST"], text="Costco's membership renewal rate stayed above 90% and fee income grew after the fee increase.")
    await call(server, "store_research", trader="George", topic="Copper supply squeeze",
               symbols=["COPX"], text="Copper inventories in Shanghai fell sharply while two large mines cut output guidance.")
    await call(server, "search_research", query="warehouse retailer customer loyalty")
    await call(server, "search_research", query="industrial metals shortage", symbol="COPX")

### Browser
Reads a page in a real browser, and refuses addresses that are not on the public internet.

In [ ]:
async with _own("browser_server") as server:
    print("Tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "read_page", url="https://example.com")
    await call(server, "read_page", url="http://127.0.0.1:8000/api/traders")

### Accounts and the knowledge graph
The accounts server only knows the four traders. Note that it still has `buy_shares`; the traders
never see it, because the floor filters it out and routes trades through the risk desk instead.

In [ ]:
async with _own("accounts_server") as server:
    print("Tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "get_balance", name="Warren")
    await call(server, "get_balance", name="Mallory")

async with memory_server("Warren") as server:
    print("Knowledge graph tools:", [tool.name for tool in await server.list_tools()], "\n")
    await call(server, "read_graph")

## 3. The risk desk's hard limits (free)

Two limits are plain arithmetic, enforced in code before any model is asked:
no position above 25% of the portfolio, and at least 2% kept in cash.

In [ ]:
from backend.risk import apply_limits

account = Account.get("Warren")
price = market.get_share_price("AAPL")
print(f"AAPL price {price:,.2f}; Warren has ${account.balance:,.2f}\n")

for label, side, quantity in [("A sensible buy", "buy", 3), ("A huge buy", "buy", 10_000), ("Selling shares he does not hold", "sell", 5)]:
    limits = apply_limits(account, "AAPL", side, quantity, price)
    print(f"{label}: asked {quantity}, allowed {limits.allowed}"
          f" | weight after {limits.weight_after:.1%} | cash after ${limits.cash_after:,.2f} | {limits.note or 'within limits'}")

## 4. The risk desk with its agent (paid: a few cents)

Three proposals from Warren, a patient value investor:

1. A small buy of a steady business: should be **approved**.
2. An enormous buy: the hard limit cuts it down, so it comes back **resized**.
3. A leveraged fund bought for a one-day flip: it contradicts his strategy, so the risk manager
   should **reject** it. This one is the model's judgement, so it can occasionally differ.

In [ ]:
from backend.risk import review_and_execute

proposals = [
    ("KO", "buy", 20, "Coca-Cola is a durable consumer franchise with steady cash flows, bought to hold for years."),
    ("NVDA", "buy", 10_000, "Strong long-term demand for AI chips; I want a large position."),
    ("TQQQ", "buy", 40, "Momentum is hot. I will flip this triple-leveraged fund tomorrow for a quick gain."),
]
with trace("Test drive: risk desk"):
    for symbol, side, quantity, rationale in proposals:
        result = await review_and_execute("Warren", symbol, side, quantity, rationale)
        print(f"{side} {quantity} {symbol} -> {result['decision'].upper()}"
              f" (executed {result.get('executed_quantity', 0)})\n   {result['reason']}\n")

In [ ]:
# What the dashboard's Risk desk tab will show for these.
for review in rows("SELECT symbol, side, requested, approved, decision, source FROM risk_reviews WHERE name = 'warren'"):
    print(review)
account = Account.get("Warren")
print("\nWarren now holds", account.holdings, "with", f"${account.balance:,.2f} cash")

## 5. Long-term memory: the researcher and the library (paid: a few cents)

First a researcher looks something up on the web and saves what it finds. Then a second agent,
which has **only the library and no web access**, answers from those saved notes.

In [ ]:
from backend.models import get_model
from backend.templates import researcher_instructions

MODEL = "gpt-5.4-mini"  # a small model is enough for this check

async with AsyncExitStack() as stack:
    shared = await start_shared_servers(stack)
    researcher = Agent(name="Researcher", instructions=researcher_instructions("Ray"),
                       model=get_model(MODEL), mcp_servers=[shared.library, shared.search, shared.fetch])
    with trace("Test drive: researcher"):
        result = await Runner.run(researcher, "What did Nvidia report in its most recent quarterly results? Keep it short.", max_turns=12)
display(Markdown(result.final_output))

In [ ]:
for note in rows("SELECT trader, topic, symbols, substr(text, 1, 160) AS text FROM research_notes ORDER BY created_at"):
    print(note)

In [ ]:
async with _own("library_server") as library:
    recall = Agent(name="Recall", model=get_model(MODEL), mcp_servers=[library],
                   instructions="Answer only from the research library. If it has nothing on the question, say so.")
    with trace("Test drive: recall"):
        result = await Runner.run(recall, "What do we already know about Nvidia's latest results?", max_turns=6)
display(Markdown(result.final_output))

## 6. One trader, one full session (paid: about a quarter of a round)

Cathie runs a complete session on her own model: research, proposals to the risk desk, trades, and
a closing appraisal. This takes a minute or two.

In [ ]:
from backend.tracers import LogTracer
from backend.traders import Trader

add_trace_processor(LogTracer())  # writes each step to the activity log, as the real floor does

async with AsyncExitStack() as stack:
    shared = await start_shared_servers(stack)
    await Trader("Cathie").run(shared, round_no=1)

session = rows("SELECT kind, model, round(ended_at - started_at) AS seconds, error, final_output FROM rounds WHERE name = 'cathie'")[-1]
print(f"{session['kind']} session on {session['model']}, {session['seconds']:.0f}s, error: {session['error']}\n")
display(Markdown(session["final_output"] or ""))

In [ ]:
account = Account.get("Cathie")
print("Cash", f"${account.balance:,.2f}", "| holdings", account.holdings, "\n")
print("Risk desk decisions:")
for review in rows("SELECT side, symbol, requested, approved, decision, substr(reason, 1, 110) AS reason FROM risk_reviews WHERE name = 'cathie'"):
    print("  ", review)
print("\nTools she and her researcher used:")
for tool in rows("SELECT substr(message, 18) AS tool, COUNT(*) AS times FROM logs WHERE name = 'cathie' AND type = 'function' "
                 "AND message LIKE 'Started%' GROUP BY message ORDER BY times DESC"):
    print("  ", tool)

## 7. See it in the dashboard, then clean up

To look at this test data in the dashboard, start the app on the scratch folder in a terminal:

```powershell
$env:DATA_DIR = "data_test"; $env:SCHEDULER_ENABLED = "false"
uv run uvicorn backend.api:app --port 8000
```

When you are done, delete the scratch folder by running the cell below. Your real `data/` folder
was never used.

In [ ]:
# import shutil; shutil.rmtree("data_test", ignore_errors=True); print("Removed data_test/")